# InstantID Mobile — Full Export Pipeline

Pipeline chuyển InstantID → ONNX → INT8 → OnnxStream cho mobile.

| Bước | Thời gian | Disk cần |
|------|-----------|----------|
| 0. Setup + tải models | ~15 phút | ~15 GB |
| 1. Fuse LCM-LoRA | ~5 phút | ~13 GB peak |
| 2. Export ONNX | ~30-60 phút | ~20 GB peak |
| 3. Quantize INT8 | ~15 phút | ~12 GB peak |
| 4. Test pipeline | ~5 phút | ~3 GB |

**Runtime:** T4 GPU (Colab free) — A100 nhanh hơn nhưng không bắt buộc.

**Google Drive cache:** Models + kết quả được cache trên Drive. Session mới restore trong ~1 phút thay vì tải lại 30 phút.

---
## Cell 0 — Mount Drive + Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CACHE = '/content/drive/MyDrive/instantid-mobile-cache'
!mkdir -p "{DRIVE_CACHE}"/{checkpoints,sdxl-base-lcm,onnx-int8}

# Core dependencies
!pip install -q \
    "diffusers>=0.28.0" \
    "transformers>=4.40.0" \
    "huggingface_hub>=0.23.0" \
    "accelerate>=0.30.0" \
    "safetensors" \
    "peft" \
    "onnx" \
    "onnxscript" \
    "onnxruntime" \
    "onnxslim" \
    "insightface" \
    "opencv-python" \
    "Pillow" \
    "scipy"

import torch, diffusers, onnxruntime
print(f"PyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}")
print(f"diffusers {diffusers.__version__}")
print(f"onnxruntime {onnxruntime.__version__}")
!df -h /content | tail -1

---
## Cell 1 — Clone Repo + Download Models

In [ ]:
import os, shutil
os.chdir('/content')

# Clone repo
if not os.path.exists('Instantid-mobile/.git'):
    !git clone https://github.com/Hert4/Instantid-mobile
else:
    !cd Instantid-mobile && git pull origin main

os.chdir('/content/Instantid-mobile')
print(f"Working dir: {os.getcwd()}")

In [ ]:
# ── Helper: restore từ Drive cache ──────────────────────────────────────
import os
DRIVE_CACHE = '/content/drive/MyDrive/instantid-mobile-cache'

def dir_size_mb(path):
    """Tính tổng size (MB) của 1 folder."""
    if not os.path.exists(path):
        return 0
    total = 0
    for dp, _, fns in os.walk(path):
        for f in fns:
            try:
                total += os.path.getsize(os.path.join(dp, f))
            except OSError:
                pass
    return total / 1024**2

def restore_from_drive(name, min_mb=10):
    """Copy từ Drive cache về local nếu có data."""
    drive_path = f'{DRIVE_CACHE}/{name}'
    local_path = f'/content/Instantid-mobile/{name}'
    drive_mb = dir_size_mb(drive_path)
    local_mb = dir_size_mb(local_path)
    if drive_mb > min_mb and local_mb < min_mb:
        print(f'  Restoring {name} from Drive ({drive_mb:.0f} MB)...')
        !cp -r "{drive_path}" "{local_path}"
        print(f'  ✅ Restored {name}')
        return True
    elif local_mb >= min_mb:
        print(f'  ✅ {name} already exists locally ({local_mb:.0f} MB)')
        return True
    return False

def save_to_drive(name, min_mb=10):
    """Copy local → Drive cache."""
    src = f'/content/Instantid-mobile/{name}'
    dst = f'{DRIVE_CACHE}/{name}'
    src_mb = dir_size_mb(src)
    dst_mb = dir_size_mb(dst)
    if src_mb < min_mb:
        print(f'  [skip] {name} too small ({src_mb:.0f} MB) — probably incomplete')
        return
    if abs(src_mb - dst_mb) < min_mb:
        print(f'  [skip] {name} already cached on Drive ({dst_mb:.0f} MB)')
        return
    print(f'  Saving {name} to Drive ({src_mb:.0f} MB)...')
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    !cp -r "{src}" "{dst}"
    print(f'  ✅ Cached {name}')

print('Helpers loaded ✅')

In [ ]:
# ── Download checkpoints ────────────────────────────────────────────────
os.chdir('/content/Instantid-mobile')
os.makedirs('checkpoints/ControlNetModel', exist_ok=True)
os.makedirs('models/antelopev2', exist_ok=True)

print('=== Checkpoints ===')
if not restore_from_drive('checkpoints', min_mb=100):
    print('  Downloading IP-Adapter + ControlNet + LCM-LoRA...')
    from huggingface_hub import hf_hub_download, snapshot_download

    # IP-Adapter (~1.6 GB)
    hf_hub_download(
        "InstantX/InstantID", "ip-adapter.bin",
        local_dir="./checkpoints"
    )
    print('  ✅ ip-adapter.bin')

    # ControlNet IdentityNet (~2.5 GB)
    snapshot_download(
        "InstantX/InstantID",
        allow_patterns=["ControlNetModel/*"],
        local_dir="./checkpoints"
    )
    print('  ✅ ControlNetModel/')

    # LCM-LoRA weights (~1.6 GB)
    hf_hub_download(
        "latent-consistency/lcm-lora-sdxl",
        "pytorch_lora_weights.safetensors",
        local_dir="./checkpoints"
    )
    print('  ✅ pytorch_lora_weights.safetensors')

    save_to_drive('checkpoints')

# InsightFace (nếu chưa có)
print('\n=== InsightFace ===')
if not os.path.exists('models/antelopev2/scrfd_10g_bnkps.onnx'):
    print('  Downloading InsightFace antelopev2...')
    from huggingface_hub import hf_hub_download
    for f in ['1k3d68.onnx', '2d106det.onnx', 'genderage.onnx',
              'glintr100.onnx', 'scrfd_10g_bnkps.onnx']:
        hf_hub_download('LPDoctor/insightface', f'antelopev2/{f}',
                        local_dir='./models')
    print('  ✅ InsightFace models')
else:
    print('  ✅ Already exists')

# Verify
print('\n=== Verification ===')
for p in ['checkpoints/ip-adapter.bin',
          'checkpoints/ControlNetModel/config.json',
          'checkpoints/pytorch_lora_weights.safetensors',
          'models/antelopev2/scrfd_10g_bnkps.onnx']:
    ok = '✅' if os.path.exists(p) else '❌ MISSING'
    print(f'  {ok}  {p}')

!df -h /content | tail -1

---
## Cell 2 — Fuse LCM-LoRA (SDXL + LoRA → sdxl-base-lcm)

Merge LCM-LoRA vào SDXL UNet → inference chỉ cần 4-8 steps thay vì 20-50.

`--delete_source` xóa sdxl-base TRƯỚC KHI save → giải phóng ~6 GB.

In [ ]:
os.chdir('/content/Instantid-mobile')

if os.path.exists('sdxl-base-lcm/model_index.json'):
    print('✅ sdxl-base-lcm đã tồn tại — skip')
elif restore_from_drive('sdxl-base-lcm', min_mb=1000):
    assert os.path.exists('sdxl-base-lcm/model_index.json'), 'Restore failed!'
else:
    # Cần download SDXL base trước
    if not os.path.exists('sdxl-base/model_index.json'):
        print('Downloading SDXL base (safetensors only, ~6.5 GB)...')
        from huggingface_hub import snapshot_download
        snapshot_download(
            "stabilityai/stable-diffusion-xl-base-1.0",
            local_dir="./sdxl-base",
            # CHỈ tải safetensors — bỏ bin/ckpt/flax/onnx = tiết kiệm ~30 GB
            ignore_patterns=["*.bin", "*.ckpt", "flax_model*", "tf_model*",
                             "*.onnx*", "*.msgpack", "*.xml", "*.pb"],
        )
        print(f'✅ Downloaded SDXL base')
    else:
        print(f'✅ sdxl-base already exists')

    !du -sh sdxl-base
    !df -h /content | tail -1

    # Fuse
    print('\nFusing LCM-LoRA...')
    !python scripts/fuse_lcm_lora.py --device cuda --delete_source

    assert os.path.exists('sdxl-base-lcm/model_index.json'), \
        'Fuse FAILED! Check errors above.'

    # Cache kết quả
    save_to_drive('sdxl-base-lcm', min_mb=1000)

print(f'\n✅ Fuse OK! sdxl-base-lcm = {dir_size_mb("sdxl-base-lcm"):.0f} MB')

In [ ]:
# Cleanup: xóa sdxl-base gốc + HF cache
import shutil
for path in ['./sdxl-base', os.path.expanduser('~/.cache/huggingface/hub')]:
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)
        print(f'Freed {path}')

!df -h /content | tail -1

---
## Cell 3 — Export ONNX

Export từng component riêng để tránh OOM.

Model được cast sang **float32 CPU** trước khi export ONNX (float16 exporter tạo file hỏng).

Quantize INT8 ở bước sau sẽ giảm size ~75%.

In [ ]:
os.chdir('/content/Instantid-mobile')
os.makedirs('onnx', exist_ok=True)

# Expected sizes (float32 export, MB)
EXPECTED_SIZES = {
    'ip_adapter':     ('onnx/ip_adapter/model.onnx',     50),
    'controlnet':     ('onnx/controlnet/model.onnx',     2000),
    'text_encoder':   ('onnx/text_encoder/model.onnx',   400),
    'text_encoder_2': ('onnx/text_encoder_2/model.onnx', 700),
    'vae':            ('onnx/vae_decoder/model.onnx',    100),
    'unet':           ('onnx/unet/model.onnx',           4000),
}

def verify_onnx(name):
    """Kiểm tra ONNX file tồn tại và đủ lớn."""
    path, min_mb = EXPECTED_SIZES[name]
    if not os.path.exists(path):
        return False
    actual_mb = os.path.getsize(path) / 1024**2
    if actual_mb < min_mb:
        print(f'  ⚠️  {name}: {actual_mb:.0f} MB < {min_mb} MB — file hỏng, sẽ re-export')
        os.remove(path)
        return False
    print(f'  ✅ {name}: {actual_mb:.0f} MB')
    return True

print('=== Current ONNX files ===')
for name in EXPECTED_SIZES:
    verify_onnx(name)

In [ ]:
# ── Export IP-Adapter + ControlNet (không cần load SDXL pipeline) ──────
os.chdir('/content/Instantid-mobile')
import gc, torch

if not verify_onnx('ip_adapter'):
    print('\n--- Exporting IP-Adapter ---')
    !python scripts/export_all.py --only ip_adapter
    assert verify_onnx('ip_adapter'), 'IP-Adapter export FAILED!'

if not verify_onnx('controlnet'):
    print('\n--- Exporting ControlNet ---')
    !python scripts/export_all.py --only controlnet
    assert verify_onnx('controlnet'), 'ControlNet export FAILED!'

gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
!df -h /content | tail -1

In [ ]:
# ── Export Text Encoders + VAE + UNet (cần load SDXL pipeline) ─────────
os.chdir('/content/Instantid-mobile')

assert os.path.exists('sdxl-base-lcm/model_index.json'), \
    'sdxl-base-lcm MISSING! Chạy lại Cell 2 (Fuse).'

for component in ['text_encoders', 'vae', 'unet']:
    # Map component name → verification key
    verify_keys = {
        'text_encoders': ['text_encoder', 'text_encoder_2'],
        'vae': ['vae'],
        'unet': ['unet'],
    }
    all_ok = all(verify_onnx(k) for k in verify_keys[component])
    if not all_ok:
        print(f'\n--- Exporting {component} ---')
        !python scripts/export_all.py --only {component}
        for k in verify_keys[component]:
            assert verify_onnx(k), f'{k} export FAILED! Check errors above.'

# Tổng kết
print('\n=== All ONNX exports ===')
total_mb = 0
for name, (path, _) in EXPECTED_SIZES.items():
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1024**2
        total_mb += mb
        print(f'  {name:20s} {mb:>8.0f} MB')
print(f'  {"TOTAL":20s} {total_mb:>8.0f} MB ({total_mb/1024:.1f} GB)')
!df -h /content | tail -1

In [ ]:
# Cleanup: xóa sdxl-base-lcm (đã export hết sang ONNX)
import shutil
if os.path.exists('sdxl-base-lcm'):
    shutil.rmtree('sdxl-base-lcm', ignore_errors=True)
    print('Freed sdxl-base-lcm')

# Xóa HF cache
hf_cache = os.path.expanduser('~/.cache/huggingface/hub')
if os.path.exists(hf_cache):
    shutil.rmtree(hf_cache, ignore_errors=True)
    print('Freed HF cache')

!df -h /content | tail -1

---
## Cell 4 — Quantize INT8

Dynamic INT8 quantization: weights INT8 offline, activations FP32 at runtime.

~75% size reduction. VAE giữ FP16 (nhạy cảm với quantization).

`--skip_slim` bỏ qua onnxslim graph optimization (OOM trên Colab với models lớn).

In [ ]:
os.chdir('/content/Instantid-mobile')

!python scripts/quantize_all.py --skip_slim

In [ ]:
# Verify quantized files
print('=== Quantized ONNX files ===')
total_mb = 0
for root, _, files in os.walk('onnx-int8'):
    for f in sorted(files):
        if f.endswith('.onnx'):
            p = os.path.join(root, f)
            mb = os.path.getsize(p) / 1024**2
            total_mb += mb
            rel = os.path.relpath(p, 'onnx-int8')
            print(f'  {rel:40s} {mb:>8.0f} MB')
print(f'  {"TOTAL":40s} {total_mb:>8.0f} MB ({total_mb/1024:.1f} GB)')

# Xóa ONNX float32 gốc (~9-20 GB)
import shutil
if os.path.exists('onnx'):
    shutil.rmtree('onnx', ignore_errors=True)
    print('\nFreed onnx/ (float32 originals)')

!df -h /content | tail -1

---
## Cell 5 — Test Pipeline

Chạy inference hoàn chỉnh với ONNX models để verify kết quả trước khi chuyển mobile.

In [ ]:
os.chdir('/content/Instantid-mobile')

# Upload face image hoặc dùng sample
import urllib.request
os.makedirs('test_images', exist_ok=True)

FACE_IMAGE = 'test_images/face.jpg'
if not os.path.exists(FACE_IMAGE):
    # Bạn có thể upload ảnh riêng:
    # from google.colab import files
    # uploaded = files.upload()
    # FACE_IMAGE = list(uploaded.keys())[0]
    print('⚠️  Chưa có face image!')
    print('Upload bằng cách uncomment code trên, hoặc:')
    print('  from google.colab import files')
    print('  uploaded = files.upload()')
else:
    print(f'Using: {FACE_IMAGE}')

In [ ]:
# Chạy test pipeline
os.chdir('/content/Instantid-mobile')

FACE_IMAGE = 'test_images/face.jpg'  # ← đổi path nếu cần

if os.path.exists(FACE_IMAGE):
    !python scripts/test_onnx_pipeline.py \
        --face_image {FACE_IMAGE} \
        --onnx_dir ./onnx-int8 \
        --insightface_dir ./models/antelopev2 \
        --prompt "portrait photo of a person, professional lighting, 4k, detailed" \
        --steps 4 \
        --seed 42 \
        --output ./output/test_result.png \
        --verbose

    # Hiển thị kết quả
    if os.path.exists('./output/test_result.png'):
        from IPython.display import Image, display
        display(Image('./output/test_result.png', width=512))
        print('✅ Pipeline test passed!')
    else:
        print('❌ Output image not found — check errors above')
else:
    print('⚠️  Skip test — upload face image trước (cell trên)')

---
## Cell 6 — Convert cho OnnxStream (Mobile)

In [ ]:
os.chdir('/content/Instantid-mobile')

if os.path.exists('scripts/convert_for_onnxstream.py'):
    !python scripts/convert_for_onnxstream.py
    !du -sh onnxstream-models/ 2>/dev/null || echo 'No output generated'
else:
    print('convert_for_onnxstream.py chưa có — dùng onnx-int8/ trực tiếp')
    print('Copy onnx-int8/ sang mobile project.')

---
## Cell 7 — Save kết quả lên Google Drive

In [ ]:
os.chdir('/content/Instantid-mobile')
DRIVE_CACHE = '/content/drive/MyDrive/instantid-mobile-cache'

# Save quantized models
save_to_drive('onnx-int8', min_mb=50)

# Save onnxstream models (nếu có)
if os.path.exists('onnxstream-models'):
    save_to_drive('onnxstream-models', min_mb=50)

# Save test output
if os.path.exists('output/test_result.png'):
    os.makedirs(f'{DRIVE_CACHE}/output', exist_ok=True)
    !cp output/test_result.png "{DRIVE_CACHE}/output/"
    print('  ✅ Saved test_result.png')

print(f'\n✅ Tất cả đã lưu tại: {DRIVE_CACHE}/')
!du -sh "{DRIVE_CACHE}"/* 2>/dev/null

In [ ]:
# (Optional) Download zip về máy local
# import shutil
# src = 'onnx-int8'  # hoặc 'onnxstream-models'
# shutil.make_archive(src, 'zip', src)
# from google.colab import files
# files.download(f'{src}.zip')

---
## Troubleshooting

| Lỗi | Giải pháp |
|------|----------|
| `ONNX file quá nhỏ (< 100 MB cho UNet)` | Bug float16 export — đã fix. Pull code mới: `git pull origin main` |
| `onnxslim treo / OOM` | Dùng `--skip_slim` — đã mặc định trong notebook |
| `sdxl-base-lcm MISSING` | Chạy lại Cell 2 (Fuse). Nếu đã bị xóa bởi `--delete_source`, cần tải lại sdxl-base |
| `RuntimeError: dtype mismatch Half/Float` | Pull code mới — fix trong `attention_processor.py` |
| `Disk full` | Xóa `~/.cache/huggingface/hub` + restart runtime |
| `Session timeout` | Mọi thứ đã cache trên Drive — chạy lại từ Cell 0, skip tự động |